# Networks and the Internet

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Follow a web request through DNS, TCP and HTTP, and choose TCP or UDP for an application
- Check that an IPv4 address is valid and read it as one 32-bit number
- Find the network, broadcast and host addresses of an address with a prefix
- Choose the next hop for a packet by longest prefix match, as a router does

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## In short

The internet never sends a file in one piece. It cuts data into **packets** of about 1500 bytes, each with a **header** (where it comes from and where it goes) and a piece of the data. **Routers** pass each packet one hop closer to its destination; packets can take different routes and arrive out of order.

**Cutting a file into packets:**

In [ ]:
int payload = 1460; // data bytes in a typical 1500-byte packet, after 40 bytes of headers
for (long size : new long[] {1_000, 100_000, 5_000_000}) {
    long packets = (size + payload - 1) / payload; // rounded up
    IO.println(String.format("%,10d bytes: %,6d packets", size, packets));
}

Every device has an **IP address**: 4 bytes written like `192.168.1.130`, which to the computer is one 32-bit number. A **prefix** such as `/26` says how many of those bits name the **network**; the rest number the **hosts** in it.

### ✏️ Exercise: Is it an octet?

Each part of an IPv4 address is a number from 0 to 255, written with 1 to 3 digits. Check one part.

In [ ]:
boolean isOctet(String part) {
    // TODO: check the length, that every char is a digit, and the range
    return false;
}

void main() {
    check("192", isOctet("192"), true);
    check("256", isOctet("256"), false);
    check("1a", isOctet("1a"), false);
    check("empty", isOctet(""), false);
}

<details>
<summary>Show an answer</summary>

```java
boolean isOctet(String part) {
    if (part.isEmpty() || part.length() > 3) {
        return false;
    }
    for (char c : part.toCharArray()) {
        if (c < '0' || c > '9') {
            return false;
        }
    }
    return Integer.parseInt(part) <= 255;
}

void main() {
    check("192", isOctet("192"), true);
    check("256", isOctet("256"), false);
    check("1a", isOctet("1a"), false);
    check("empty", isOctet(""), false);
}
```

</details>

![The address 192.168.1.130/26 split into network and host bits](prefix.svg)

#### 🎞️ Splitting an address with a prefix

![For 192.168.1.130/26 the first two bits of the last byte name the network: network .128, broadcast .191, devices .129 to .190](subnet-steps.svg)

All host bits 0 is the **network address**, all 1 is the **broadcast address**, and the addresses in between are for devices. A router chooses where to send a packet by the **longest prefix** in its table that matches the destination.

**How many addresses does each prefix give?**

In [ ]:
for (int prefix = 22; prefix <= 30; prefix++) {
    long addresses = 1L << (32 - prefix);
    IO.println(String.format("/%d: %,5d addresses, %,5d for devices", prefix, addresses, addresses - 2));
}

### ✏️ Exercise: Devices in a network

A prefix /p leaves 32 − p bits for hosts, so 2³²⁻ᵖ addresses. Two of them (the network and the broadcast address) can't be used by devices.

In [ ]:
long devices(int prefix) {
    // TODO: count the addresses with a shift, minus 2
    return 0;
}

void main() {
    check("/24", devices(24), 254L);
    check("/26", devices(26), 62L);
    check("/30", devices(30), 2L);
}

<details>
<summary>Show an answer</summary>

```java
long devices(int prefix) {
    return (1L << (32 - prefix)) - 2;
}

void main() {
    check("/24", devices(24), 254L);
    check("/26", devices(26), 62L);
    check("/30", devices(30), 2L);
}
```

</details>

On top of IP, **TCP** numbers the data, confirms what arrived and resends what was lost, so pages and files arrive complete and in order. **UDP** skips all that, which suits video calls and games, where a late packet is useless anyway. Before connecting, your computer asks **DNS** for the address of a name like `example.com`. Then the browser opens a TCP connection and sends an **HTTP** request: `GET /index.html HTTP/1.1`.

## Readings

- [The Internet](https://www.youtube.com/watch?v=AEaKrq3SpW8) (Crash Course Computer Science #29, 12 min): How a message becomes packets, how routers pass them along hop by hop, why TCP numbers and acknowledges packets while UDP doesn't, and what DNS does before any of it.
- [How DNS Works](https://howdns.works/) (DNSimple (a comic in 6 short episodes), 15 min): Follow one name from the browser cache to the resolver, then the root, TLD and authoritative servers, and notice where each answer is cached for next time.
- [Subnet Cheat Sheet](https://www.freecodecamp.org/news/subnet-cheat-sheet-24-subnet-mask-30-26-27-29-and-other-ip-address-cidr-network-references/) (freeCodeCamp, 9 min): How a prefix such as /26 splits an address into network and host bits, how a mask in binary marks the network part, and the table of prefixes with their usable addresses (/26 has 62). The wildcard masks and the /31 special case are more than you need.

## Check yourself

1. Why does a video call use UDP while a download uses TCP?
2. What are the network and broadcast addresses of `10.20.30.40/24`?
3. A router has routes for `10.0.0.0/8` and `10.1.0.0/16`. Where does it send a packet for `10.1.2.3`, and why?

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Valid IPv4 Address** · Easy
- **Read an HTTP Request** · Easy
- **Subnet Calculator** · Medium
- **Routing Table** · Hard
- Test: **Packet Inspector** · Hard